# Biomass LA2 product Congo basin vs CCI Biomass

Objetive: 
With the map provide in the folder: BIOMASS-Coding2026/Biomass_AGBD_Congo_Basin/agbd_map_L2Afullextent_congo_basin.tif extract information available from CCI biomass

### 1. Packages and Libraries

In [ ]:
%pip install aiohttp requests

from shapely.geometry import Polygon, box
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

## 2. Open AGB LA2 Congo map

In [ ]:
# Path to your GeoTIFF file
tif_path = r'Examples_and_Data/Workshops/BIOMASS-Coding2026/Biomass_AGBD_Congo_Basin/agbd_map_L2Afullextent_congo_basin.tif'

with rio.open(tif_path) as src:
    print("File:", src.name)
    print("Raster dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Coordinate reference system:", src.crs)
    print("Pixel resolution:", src.res)
    print("No-data value:", src.nodata)
    print("Band descriptions:", src.descriptions)
    print("Bounds:", src.bounds)

agb_bbox = [coord for coord in src.bounds]
agb_bbox

## 3. Query the ESA MAAP STAC catalog

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

### 3.1.  Select a collection and narrow down the search
Using the bounding box of the AGB map from Congo get the CCI Biomass items

In [ ]:
search = catalog.search(
    collections = ["CCIBiomassV5.01"],
    datetime = ['2021-01-01T00:00:00Z','2021-12-31T00:00:00Z'],
    bbox = agb_bbox,
    method = "GET",
    max_items = 50,  
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

Generate the gdf with the information of CCI biomass available

In [ ]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    if "L4-AGB-MERGED" in it.id and "SD_MERGED" not in it.id:
        props = it.properties or {}
        features.append({
            "type": "Feature",
            "geometry": it.geometry,
            "properties": {
                "product_id": it.id,
                "start_datetime": props.get("start_datetime"),
                "end_datetime": props.get("end_datetime"),
                "item": it
            },
        })


gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
# gdf = gdf[["index", "product_id", "start_datetime", "end_datetime", "geometry"]]
gdf = gdf[["product_id", "start_datetime", "end_datetime", "geometry","item"]]
gdf.head(len(items))

### 4. Save the CCI biomass information available

In [ ]:
# Save the gdf
#gdf.to_csv(r'CCI_biomass_products_Congo.csv', index=False)
gdf.__class__ = gpd.GeoDataFrame
gdf.set_geometry('geometry')
gdf.to_file(r'CCI_biomass_products_Congo.shp', driver='ESRI Shapefile')
print(f'Ruta de guardado y nombre de archivo: {r'CCI_biomass_products_Congo.shp'} \n CRS: {gdf.crs} \n Número de elementos: {len(gdf)}')

Visualize the CCI biomass footprints

In [ ]:
# Get CCI biomass bounding box
gdf_bbox = gdf['geometry'].total_bounds
#gdf_bbox = gdf.loc[19,'geometry'].bounds
gdf_bbox = [float(coord) for coord in gdf_bbox]
minx, miny, maxx, maxy = gdf_bbox
# geom = box(minx, miny, maxx, maxy)
print(minx, miny, maxx, maxy)

center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=3)

def style_biomass(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.10} # orange

GeoJson(
gdf.iloc[:,:-1].__geo_interface__,
name=f"CCI BIOMASS ({len(gdf)})",
tooltip=folium.GeoJsonTooltip(fields=["product_id"]),
style_function=style_biomass,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m